In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
def cosine_similarity(a, b):
    return (np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
 

def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    result = []
    for start in range(0, len(words), chunk_size - overlap):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        result.append({"chunk_id": len(result), "text": chunk})
        if end >= len(words):
            break
    return result

def retrieve(query, k=3):
    query_vec = model.encode(query)
    scores = []
    for ch_v, ch in zip(chunk_vector, chunks):
        sim = float(cosine_similarity(ch_v, query_vec))
        scores.append({"id": ch["chunk_id"], "score": sim, "chunk": ch})
        
    s_scores = sorted(scores, key=lambda x: x["score"], reverse=True)
    return s_scores[:k]

def expand_with_neighbors(results, window=1):
    ids = set()
    for r in results:
        center = r["id"]
        for neighbor in range(center - window, center + window + 1):
            if 0 <= neighbor < len(chunks):
                ids.add(neighbor)
    return [chunks[i] for i in sorted(ids)]

def find_chunk(keyword):
    return [c["chunk_id"] for c in chunks if keyword.lower() in c["text"].lower()]

In [ ]:
with open("doc.txt", encoding= "utf-8") as f:
    text = f.read()

In [ ]:
chunks = chunk_text(text)
print(f"{len(chunks)} Chunks")

In [ ]:
chunk_vector = model.encode([c["text"] for c in chunks])

In [ ]:
find_chunk("failure")

In [ ]:
eval_set = [
    {"query": "how much protein should I eat per day?",                 "gold": [27, 28]},
    {"query": "can I bring a friend with me to the gym?",               "gold": [48]},

    {"query": "why are my legs still sore two days after squats?",      "gold": [36, 37]},
    {"query": "my weight isn't changing, how do I know if I'm losing fat?", "gold": [44]},
    {"query": "how many days off should a beginner take each week?",    "gold": [36]},

    {"query": "can I use my student discount on the Titan plan?",       "gold": [47]},
    {"query": "is it fine to do HIIT every day?",                       "gold": [24]},
    {"query": "should I hold long stretches before lifting heavy?",     "gold": [21]},

    {"query": "what does error IH-409 mean?",                           "gold": [51]},

    {"query": "is creatine bad for my kidneys?",                        "gold": [32]},
]

In [ ]:
def recall_at_k(eval_set, k=3):    
    count = 0
    for q in eval_set:
        retrieved_ids = [r["id"] for r in retrieve(q["query"], k=k)]
        if set(q["gold"]) & set(retrieved_ids):
            count += 1
        else:
            print(f"MISS: {q['query']}  gold={q['gold']}  got={retrieved_ids}")

    print(f"recall@{k} = {count / len(eval_set):.2f}")

recall_at_k(eval_set, k=3)

In [ ]:
while True:
    query = input("\nQuestion (or 'q' to quit): ")
    if query == 'q':
        break
    for r in retrieve(query):
        print(f"\n[{r['id']}] {r['score']:.3f}\n{r['chunk']['text'][:300]}...")